In [ ]:
!nvidia-smi

FILE = "multiheadatten/1st.cu"   # change this to the kernel in kernels/ you want to run

!rm -rf /content/repo && git clone -q --depth 1 https://github.com/Aakash-Rajput-2024/cuda-training.git /content/repo

import os, subprocess
src = "/content/repo/kernels/" + FILE
exe = "/tmp/" + os.path.splitext(FILE)[0]
os.makedirs(os.path.dirname(exe), exist_ok=True)

if not os.path.exists(src):
    print("Not found:", FILE)
    print("Available:", os.listdir("/content/repo/kernels"))
else:
    b = subprocess.run(["nvcc", "-O2", "-arch=sm_75", "-lineinfo", src, "-o", exe],
                       capture_output=True, text=True)
    if b.returncode:
        print(b.stderr)
    else:
        print(subprocess.run([exe], capture_output=True, text=True).stdout, end="")

### Benchmark Multi-Head Attention Kernels & Correctness Checker
Runs all kernels in `kernels/multiheadatten/` side-by-side with an **output correctness checker** and **max size 14 GB memory stress test**.

In [ ]:
import os, subprocess, re
import pandas as pd

# ==============================================================================
# Benchmark Configuration: Max Size Considering ~14 GB RAM / VRAM
# ==============================================================================
# Set STRESS_TEST = True to run the 14 GB scale stress test (~12.1 GB GPU footprint)
# Set STRESS_TEST = False for a fast iteration test (takes ~1-2 seconds)
STRESS_TEST = True

if STRESS_TEST:
    # Max configuration for ~14 GB memory:
    # Q, K, V, Out: 4 * (40960 * 8192 * 4 bytes) = 5.37 GB
    # Attention Scores buffer: 40960 * 40960 * 4 bytes = 6.71 GB
    # Total GPU Memory Footprint: ~12.08 GB (safe maximum for 14-16 GB GPU/RAM)
    N_VAL   = 40960   # Sequence length (40k tokens)
    D_MODEL = 8192    # Hidden dimension
    HEADS   = 32      # Number of heads (dk = 256)
else:
    N_VAL   = 2048    # Sequence length
    D_MODEL = 2048    # Hidden dimension
    HEADS   = 16      # Heads (dk = 128)

total_mem_gb = (4.0 * N_VAL * D_MODEL * 4 + N_VAL * N_VAL * 4) / (1024**3)
mode_str = "14 GB STRESS TEST" if STRESS_TEST else "FAST TEST"
print(f"[MHA Configuration] Mode: {mode_str}")
print(f"  Sequence Length N = {N_VAL}, Hidden Dim d_model = {D_MODEL}, Heads = {HEADS}")
print(f"  Total GPU Memory  = {total_mem_gb:.2f} GB\n")

mha_dir = "/content/repo/kernels/multiheadatten"
files = sorted([f for f in os.listdir(mha_dir) if f.endswith(".cu")]) if os.path.exists(mha_dir) else []

results = []
baseline_lat = None

for f in files:
    src = os.path.join(mha_dir, f)
    exe = "/tmp/mha_" + os.path.splitext(f)[0]
    
    # Compile with optimization
    b = subprocess.run(["nvcc", "-O2", "-arch=sm_75", "-lineinfo", src, "-o", exe],
                       capture_output=True, text=True)
    if b.returncode:
        print(f"[Compile Error] {f}:\n", b.stderr)
        results.append({
            "Kernel": f,
            "Output Correct?": "COMPILE ERROR ❌",
            "Max Diff": "-",
            "Latency (ms)": None,
            "Throughput (GFLOP/s)": None,
            "Speedup": "-"
        })
        continue
    
    # Run executable with stress test parameters: ./exe [N] [d_model] [heads]
    print(f"--> Running {f} with N={N_VAL}, d_model={D_MODEL}, h={HEADS}...")
    run_res = subprocess.run([exe, str(N_VAL), str(D_MODEL), str(HEADS)],
                             capture_output=True, text=True)
    out = run_res.stdout
    print(out)
    if run_res.stderr:
        print("[Stderr]:", run_res.stderr)
    
    # Output Correctness Checker
    is_pass = ("Verification : PASS" in out) or ("[PASS]" in out)
    is_fail = ("Verification : FAIL" in out) or ("[FAIL]" in out) or (run_res.returncode != 0)
    has_nan_inf = ("NaN" in out) or ("Inf" in out)
    
    diff_match = re.search(r"max diff(?:erence)?:\s*([0-9\.eE\-\+]+)", out, re.IGNORECASE)
    max_diff_str = diff_match.group(1) if diff_match else ("0.0" if is_pass else "FAILED")
    
    if is_pass and not is_fail and not has_nan_inf:
        checker_status = "PASS ✓"
    else:
        checker_status = "FAIL ❌"
        print(f"⚠️  ALERT: Output check FAILED for {f}!")
    
    lat_match = re.search(r"Avg Latency\s*:\s*([\d\.]+)\s*ms", out)
    gflop_match = re.search(r"Throughput\s*:\s*([\d\.]+)\s*GFLOP/s", out)
    lat = float(lat_match.group(1)) if lat_match else None
    gflops = float(gflop_match.group(1)) if gflop_match else None
    
    if baseline_lat is None and lat and checker_status == "PASS ✓":
        baseline_lat = lat
    speedup = (baseline_lat / lat) if (baseline_lat and lat) else 1.0
    
    results.append({
        "Kernel": f,
        "Output Correct?": checker_status,
        "Max Diff": max_diff_str,
        "Latency (ms)": lat,
        "Throughput (GFLOP/s)": gflops,
        "Speedup": f"{speedup:.2f}x" if lat else "-"
    })

if results:
    df = pd.DataFrame(results)
    print("\n================ Multi-Head Attention Benchmark Summary ================")
    print(df.to_string(index=False))


### Plot MHA Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

if results and any(r["Latency (ms)"] for r in results):
    valid = [r for r in results if r["Latency (ms)"] is not None]
    names = [r["Kernel"] for r in valid]
    lats = [r["Latency (ms)"] for r in valid]
    speedups = [float(r["Speedup"].replace("x", "")) if "x" in str(r["Speedup"]) else 1.0 for r in valid]
    colors = ["#55A868" if "PASS" in r["Output Correct?"] else "#C44E52" for r in valid]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    
    # Latency
    bars1 = ax1.bar(names, lats, color="#4C72B0", width=0.4)
    ax1.set_title(f"Latency (ms) - Lower is better\n(N={N_VAL}, d={D_MODEL}, h={HEADS})", fontsize=11)
    ax1.set_ylabel("ms")
    ax1.grid(axis="y", linestyle="--", alpha=0.5)
    for b in bars1:
        ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.2f} ms", ha="center", va="bottom", fontsize=9)
    
    # Speedup
    bars2 = ax2.bar(names, speedups, color=colors, width=0.4)
    ax2.axhline(1.0, color="gray", linestyle="--", label="1.0x (1st.cu)")
    ax2.set_title("Speedup vs Baseline - Higher is better\n(Green = Output Verified)", fontsize=11)
    ax2.set_ylabel("Speedup Multiplier")
    ax2.grid(axis="y", linestyle="--", alpha=0.5)
    ax2.legend()
    for b in bars2:
        ax2.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.2f}x", ha="center", va="bottom", fontsize=9)
    
    plt.tight_layout()
    plt.show()
